# Exercise Classifier Playground

This notebook is a cleaned training/evaluation playground aligned with `utils.py`.

In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import sys
sys.path.insert(0, str(Path('.').resolve()))
from utils import normalize_skeleton_for_classifier, extract_features_from_skeleton


In [ ]:
DATASET_DIRS = [
    Path('../datasets/mm-fit/skeleton/json_raw'),
    Path('../datasets/youtube/skeleton_json'),
]

LABEL_MAP = {
    'bicep_curls': 0,
    'lateral_shoulder_raises': 1,
    'null': 2,
    'shoulder_press': 3,
    'tricep_extensions': 4,
    'front_raise': 5,
}


In [ ]:
def get_subject_id(filename: str) -> str:
    return filename.split('_')[0]

def skeleton_to_numpy(frames):
    arr = np.zeros((len(frames), 33, 4), dtype=np.float32)
    for i, frame in enumerate(frames):
        lms = frame.get('landmarks', []) if isinstance(frame, dict) else frame
        for j, lm in enumerate(lms[:33]):
            arr[i, j, 0] = lm['x']
            arr[i, j, 1] = lm['y']
            arr[i, j, 2] = lm['z']
            arr[i, j, 3] = lm.get('visibility', 1.0)
    return arr

def build_feature_table(dataset_dirs, smoothing_window=5):
    clips = []
    for root in dataset_dirs:
        if not root.exists():
            continue
        for class_dir in sorted([d for d in root.iterdir() if d.is_dir()]):
            class_name = class_dir.name
            if class_name not in LABEL_MAP:
                continue
            label = LABEL_MAP[class_name]
            for json_file in sorted(class_dir.glob('*.json')):
                with json_file.open('r') as f:
                    frames = json.load(f)
                if not frames:
                    continue

                arr = skeleton_to_numpy(frames)
                if smoothing_window > 0:
                    for j in range(33):
                        for k in range(3):
                            arr[:, j, k] = (
                                pd.Series(arr[:, j, k])
                                .rolling(window=smoothing_window, center=True, min_periods=1)
                                .mean()
                                .to_numpy()
                            )

                normalized_arr = normalize_skeleton_for_classifier(arr)
                features = extract_features_from_skeleton(normalized_arr)
                features['label'] = label
                features['subject'] = get_subject_id(json_file.name)
                features['file'] = json_file.name
                features['frame'] = np.arange(len(features))
                clips.append(features)

    if not clips:
        return pd.DataFrame()
    return pd.concat(clips, ignore_index=True)


In [ ]:
df = build_feature_table(DATASET_DIRS, smoothing_window=5)
print('rows:', len(df))
print(df['label'].value_counts(dropna=False))
df.head()


In [ ]:
train_subjects = [
    'tony', 'Jab1Mo-WR9o', 'ABSUGEVlGKA', 'H0sEj8Q6u7U',
    'w01', 'w02', 'w13', 'w14', 'w15', 'w16', 'w17', 'w18', 'w20'
]
test_subjects = ['w00', 'w19', 'Jpxc0TUr9BI', 'XPU1Xl6nZXI']

feature_cols = [c for c in df.columns if c not in ['label', 'subject', 'file', 'frame']]
train_df = df[df['subject'].isin(train_subjects)]
test_df = df[df['subject'].isin(test_subjects)]

X_train = train_df[feature_cols]
y_train = train_df['label']
X_test = test_df[feature_cols]
y_test = test_df['label']

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print('Accuracy:', accuracy_score(y_test, y_pred))
print('\nConfusion Matrix:\n', confusion_matrix(y_test, y_pred))
print('\nClassification Report:\n', classification_report(y_test, y_pred))


In [ ]:
OUT_MODEL = Path('../models/exercise_classifier_rf.joblib')
OUT_MODEL.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(clf, OUT_MODEL)
print('saved model to', OUT_MODEL)
